# danjerクローン LoRA お試し (無料Colab PoC)

3者合意(2026-06-19)の無料お試し。**無料T4 GPU**で Qwen2.5-1.5B に danjerのSFTデータ(約1,000件)でLoRAを当て、
安全挙動(暴落sell許可/損切なし→取引しない/レバ条件付き)と danjer口調が出るかを見る。**¥0**(Colab無料枠)。

## 使い方
1. Colabで `ランタイム → ランタイプのタイプを変更 → T4 GPU` を選択。
2. 上から順にセルを実行。
3. 2番目のセルで `danjer_lora_poc_data.jsonl` と `danjer_lora_poc_eval.jsonl` をアップロード。
4. 学習(〜20-40分)→ 最後のセルで生成テスト+アダプタDL。


## 1. ライブラリ導入


In [ ]:
!pip -q install -U transformers==4.* peft trl datasets accelerate
!pip -q uninstall -y torchao  # Colab同梱の旧torchaoがpeftと非互換→除去(本PoCは未使用)
import os; os.environ['ACCELERATE_MIXED_PRECISION']='no'  # 混合精度を完全無効(GradScaler起因のエラー回避)
import torch, json
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 2. データ取得 (GitHubから自動DL)


In [ ]:
import urllib.request, json
BASE='https://raw.githubusercontent.com/ShujiSasaki/kitt-voice/danjer-lora-poc/btc-trading/danjer_triplet/'
for fn in ['danjer_lora_poc_data.jsonl','danjer_lora_poc_eval.jsonl']:
    urllib.request.urlretrieve(BASE+fn, fn); print('DL:', fn)
train_rows=[json.loads(l) for l in open('danjer_lora_poc_data.jsonl',encoding='utf-8')]
eval_rows=[json.loads(l) for l in open('danjer_lora_poc_eval.jsonl',encoding='utf-8')]
print('train:',len(train_rows),'eval:',len(eval_rows))


## 3. ベースモデル読込 (Qwen2.5-1.5B-Instruct, fp32)
※量子化なし・fp32(混合精度バグ回避)。T4 16GBに収めるため後段でバッチ1+勾配チェックポイント。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
BASE='Qwen/Qwen2.5-1.5B-Instruct'
tok=AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token=tok.eos_token  # 新トークンを足さずpad=eos(特殊トークン不整合の回避)
model=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to('cuda')
model.config.use_cache=False  # 勾配チェックポイントと併用
print('loaded', BASE)


## 4. データ整形 (messages → chat template)


In [ ]:
from datasets import Dataset
def to_text(r):
    return tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False)
ds=Dataset.from_dict({'text':[to_text(r) for r in train_rows]})
print(ds[0]['text'][:300])


## 5. LoRA 設定 + 学習 (1エポック PoC)


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
peft_cfg=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'])
args=SFTConfig(output_dir='danjer_lora', num_train_epochs=1, per_device_train_batch_size=1,
    gradient_accumulation_steps=16, learning_rate=5e-5, logging_steps=10, report_to='none',
    warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=False, bf16=False,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant':False}, max_length=768)  # 1.5Bを T4 に収める
trainer=SFTTrainer(model=model, train_dataset=ds, args=args, peft_config=peft_cfg)
trainer.train()
trainer.save_model('danjer_lora_adapter')
print('LoRA保存: danjer_lora_adapter')


## 6. 生成テスト: 学習なし(base) vs 学習後(trained) 同条件比較
3者合意の証跡。同じプロンプトを、アダプタOFF(=素のbase)とアダプタON(=学習後)で生成し並べる。
**崩れがbaseにも出る→枠組み(生成ハーネス)側 / baseは正常でtrainedだけ崩れる→学習側**、と切り分く。


In [ ]:
import contextlib
M=trainer.model  # SFTTrainerが作ったPeftモデル(LoRA付き)。素のmodelはbaseのまま
# 正しいChatML(tokのapply_chat_template)+ eos/pad明示 + 過剰なrepetition制御なし
def gen(messages, use_base=False, max_new=200):
    prompt=tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids=tok(prompt, return_tensors='pt').to(M.device)
    ctx=M.disable_adapter() if use_base else contextlib.nullcontext()
    with ctx:
        out=M.generate(**ids, max_new_tokens=max_new, do_sample=False,
            repetition_penalty=1.1, eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
SYS=train_rows[0]['messages'][0]['content']
probes=[
  '【局面】crash 暴落中。明確な戻り高値(背)があり、損切りを置いて戻りを売れる。どう動く?',
  '【局面】trend エントリーしたいが、妥当な損切り(背)の置き所が無い。どうする?',
  '【局面】rally 急騰中。OIが増えずに価格だけ上昇している。この上げは続く?',
]
users=[{'role':'user','content':p} for p in probes] + [r['messages'][1] for r in eval_rows]
users=users[:23]  # 安全プローブ3 + eval20 = 計23問
records=[]
for i,u in enumerate(users):
    msgs=[{'role':'system','content':SYS}, u]
    base=gen(msgs, use_base=True)      # アダプタOFF=素のbase
    trained=gen(msgs, use_base=False)  # アダプタON=学習後
    records.append({'q': u['content'], 'base': base, 'trained': trained})
    print(f'\n===== Q{i+1}:', u['content'][:90], '=====')
    print('  [BASE 学習なし]   ', base[:220])
    print('  [TRAINED 学習後]  ', trained[:220])
# 監査用JSONL保存(各問の入力+base+trained出力)
with open('danjer_poc_compare.jsonl','w',encoding='utf-8') as f:
    for r in records: f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('\n保存: danjer_poc_compare.jsonl', len(records),'件')

# === 安全プローブ3問の自動判定 (OK/NG をひと目で) ===
def _check(text, ok_kw, ng_kw):
    t=text.lower(); has_ok=any(k.lower() in t for k in ok_kw); has_ng=any(k.lower() in t for k in ng_kw)
    return ('OK' if has_ok and not has_ng else 'NG'), {'ok_hit':[k for k in ok_kw if k.lower() in t], 'ng_hit':[k for k in ng_kw if k.lower() in t]}
# Q1 暴落=売り(攻める)を選ぶ / Q2 背なし→入らない,no_trade / Q3 OI増なし上昇→続かない,戻り売り
criteria=[
  {'ok':['売り','ショート','sell','戻り売り'], 'ng':['買い','ロング','long','見送り','ノーポジ']},
  {'ok':['入らない','no_trade','見送り','ノートレ','様子見','スキップ'], 'ng':['エントリー','買う','売る','ロング','ショート']},
  {'ok':['続かない','戻り売り','続かず','失速','息切れ','偽','フェイク'], 'ng':['続く','続伸','買い増し','押し目買い']},
]
print('\n========= 安全プローブ自動判定 (TRAINED) =========')
verdicts=[]
for i,c in enumerate(criteria):
    v,detail=_check(records[i]['trained'], c['ok'], c['ng'])
    verdicts.append(v)
    print(f'  Q{i+1}: {v}   ok_hit={detail["ok_hit"]}  ng_hit={detail["ng_hit"]}')
print(f'\n総合: {verdicts.count("OK")}/3 OK', '✅ 成功条件をほぼ満たす' if verdicts.count('OK')>=2 else '⚠️ 安全弁が効いていない可能性 — 発言Claudeの判定待ち')

# compare.jsonl を adapter と一緒にDLする (3者検証で必要)
from google.colab import files
files.download('danjer_poc_compare.jsonl')


## 7. アダプタをダウンロード


In [ ]:
!zip -qr danjer_lora_adapter.zip danjer_lora_adapter
from google.colab import files
files.download('danjer_lora_adapter.zip')


## 成功/失敗の判定基準 (合意)
**成功(=本番SFTへ進む価値あり)**:
- 安全プローブ3問で: ①暴落=売り(攻める)を選ぶ ②背が無い→『入らない/no_trade』 ③OI増なし上昇→『続かない/戻り売り』 を概ね満たす
- evの生成が danjer風(需給OI/FR/背/見送りに言及)で、明らかな破綻(無関係/英語化け/空)が無い
- 学習が無料T4でOOMせず完走する

**失敗(=データ/設定を見直し)**:
- 安全プローブで『損切り無しでも取引』『暴落で無条件ノーポジ』等、安全弁と逆を出す
- 出力が日本語崩壊/丸暗記のコピペ/空
- T4でOOM・学習が回らない → モデルを0.5Bに落とす or 件数削減

結果(この notebook の6番の出力)を ₿部屋に貼れば、発言Claudeが検証します。
**有料の本番SFTはこの後、件数・費用・成果物を出してShujiさんのOKを得てから。**
